In [ ]:
# ══════════════════════════════════════════════════════
#  Fuzzy Decision Tree — Gate-level & End-to-end Evaluation
# ══════════════════════════════════════════════════════
#
# 前置条件: 先运行 D_S3_fuzzy_calibration.ipynb (或 _mild)
#           生成 output/S3_polyfit/fuzzy_params.json
#
# 评价方式:
#   1. Oracle-path: 用人工标签过滤上游，评价每个 gate 自身能力
#   2. End-to-end: 用算法预测逐层传递，评价整体性能
#
# 三个评价量:
#   - correct rate: 只在明确标签 0/1 上计算
#   - uncertain rate: membership 严格在 (0, 1) 之间
#   - mean membership: 正例/负例分开

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json as json_mod
import warnings, os

os.environ.setdefault('MPLCONFIGDIR', str(Path('.matplotlib').resolve()))
warnings.filterwarnings('ignore')

# ── 选择 variant ──
VARIANT = 'strong'   # 'strong' or 'mild'
ALPHA = 0.5

# ── 数据加载 ──
S1_DIR = Path('output/S1')
S2_DIR = Path('output/S2')
OUT_DIR = Path('output/S3_polyfit')

cases = pd.read_csv(S1_DIR / 'cases.csv', low_memory=False)
metrics = pd.read_parquet(S2_DIR / 'metrics_full.parquet')
cases['snr_float'] = cases['snr'].apply(lambda x: float(x))

pts = np.load(S1_DIR / 'scatter_points.npz')
x_all, y_all = pts['x'], pts['y']
case_idx_map = pd.Series(np.arange(len(cases)), index=cases['case_id']).to_dict()

df = cases.merge(
    metrics[['case_id', 'MIC', 'distance_correlation', 'pearson_r', 'spearman_rho']],
    on='case_id', how='inner',
)

variant_level = VARIANT
data = df[df['variant_level'] == variant_level].copy()
null = df[df['family_id'] == 'Null'].copy()
data = pd.concat([data, null], ignore_index=True)
data = data[np.isfinite(data['snr_float'])].copy()

data['abs_rho'] = data['spearman_rho'].abs()
data['abs_r'] = data['pearson_r'].abs()

FAMILIES = [f'F{i:02d}' for i in range(1, 27)]
ALL_FIDS = sorted(cases['family_id'].unique())
SHORT = {}
for fid in ALL_FIDS:
    row = cases[cases['family_id'] == fid].iloc[0]
    SHORT[fid] = str(row['family_name'])

print(f'Variant: {VARIANT}')
print(f'Total cases: {len(data):,}')

In [ ]:
# ══════════════════════════════════════════════════════
#  加载校准参数 + 定义辅助函数
# ══════════════════════════════════════════════════════

params_file = OUT_DIR / 'fuzzy_params.json'
with open(params_file) as f:
    P = json_mod.load(f)

print(f'Loaded params from {params_file}')
print(f'  Steps: {[k for k in P if k.startswith("step")]}')

# ── fuzzy membership ──
def fuzzy_mu(z, L, U):
    if U == L:
        return 1.0 if z >= U else 0.0
    return np.clip((z - L) / (U - L), 0.0, 1.0)

# ── BIC model comparison ──
def bic_compare(x, y):
    order = np.argsort(x)
    xs = x[order].astype(np.float64)
    ys = y[order].astype(np.float64)
    x_mu, x_sd = xs.mean(), max(xs.std(), 1e-10)
    y_mu, y_sd = ys.mean(), max(ys.std(), 1e-10)
    xn = (xs - x_mu) / x_sd
    yn = (ys - y_mu) / y_sd
    n = len(xn)
    ss_tot = np.sum((yn - yn.mean()) ** 2)
    bics = {}
    for deg, k in [(1, 2), (2, 3), (3, 4)]:
        coeffs = np.polyfit(xn, yn, deg)
        pred = np.polyval(coeffs, xn)
        rss = np.sum((yn - pred) ** 2)
        bics[deg] = n * np.log(rss / n + 1e-30) + k * np.log(n)
    return bics[1] - min(bics[2], bics[3])

# ── Curvature evidence ──
INTERIOR_MARGIN_S5 = 0.10
N_GRID_S5 = 200

def curvature_evidence(x, y):
    order = np.argsort(x)
    xs = np.asarray(x, dtype=float)[order]
    ys = np.asarray(y, dtype=float)[order]
    xn = (xs - xs.mean()) / max(xs.std(), 1e-10)
    yn = (ys - ys.mean()) / max(ys.std(), 1e-10)
    coeffs = np.polyfit(xn, yn, 3)
    p = np.poly1d(coeffs)
    x_lo = xn.min() + INTERIOR_MARGIN_S5 * (xn.max() - xn.min())
    x_hi = xn.max() - INTERIOR_MARGIN_S5 * (xn.max() - xn.min())
    grid = np.linspace(x_lo, x_hi, N_GRID_S5)
    f1 = np.polyder(p, 1)(grid)
    f2 = np.polyder(p, 2)(grid)
    pos_mass = np.abs(f2[f2 > 0]).sum()
    neg_mass = np.abs(f2[f2 < 0]).sum()
    total_mass = pos_mass + neg_mass
    if total_mass < 1e-12:
        q_pos = q_neg = 0.5
    else:
        q_pos = pos_mass / total_mass
        q_neg = neg_mass / total_mass
    curvature_strength = np.mean(np.abs(f2)) / (np.mean(np.abs(f1)) + 1e-12)
    f2_dominance = max(q_pos, q_neg)
    f2_dom_sign = 1 if q_pos >= q_neg else -1
    return curvature_strength, f2_dominance, f2_dom_sign

# ── 提取参数 ──
MIC_L, MIC_U = P['step1_mic']['L'], P['step1_mic']['U']
RHO_L_S2, RHO_U_S2 = P['step2_rho']['L'], P['step2_rho']['U']
R_L_S2, R_U_S2 = P['step2_r']['L'], P['step2_r']['U']
RHO_L_S3, RHO_U_S3 = P['step3_rho']['L'], P['step3_rho']['U']
R_L_S4, R_U_S4 = P['step4_r']['L'], P['step4_r']['U']
BIC_L_S4, BIC_U_S4 = P['step4_delta_bic']['L'], P['step4_delta_bic']['U']

if 'step5_curvature' in P:
    C_L = P['step5_curvature']['curvature_strength']['L']
    C_U = P['step5_curvature']['curvature_strength']['U']
    D_L = P['step5_curvature']['f2_dominance']['L_D']
    D_U = P['step5_curvature']['f2_dominance']['U_D']

# ── 提取标注分组 ──
ann = P['annotations']
groups = P['family_groups']
NO_GLOBAL = groups['no_global']
SIMPLE_FAM = groups['simple']
UNCERTAIN_FAM = groups.get('uncertain', [])
COMPLEX_FAM = groups['complex']
STRONG_MONO = groups['strong_mono']
UNCERTAIN_MONO = groups.get('uncertain_mono', [])
WEAK_MONO = groups.get('weak_mono', [])
LINEAR_FAM = groups['linear']
CURVATURE_CLASS = groups.get('curvature_class', {})

CLEAR_S1 = {k: float(v) for k, v in ann['step1']['CLEAR_SNR'].items()}
THRESH_S1 = {k: float(v) for k, v in ann['step1']['THRESHOLD_SNR'].items()}
CLEAR_S2 = {k: float(v) for k, v in ann['step2']['CLEAR_SNR'].items()}
THRESH_S2 = {k: float(v) for k, v in ann['step2']['THRESHOLD_SNR'].items()}
CLEAR_S3 = {k: float(v) for k, v in ann['step3']['CLEAR_SNR'].items()}
THRESH_S3 = {k: float(v) for k, v in ann['step3']['THRESHOLD_SNR'].items()}
CLEAR_S4 = {k: float(v) for k, v in ann['step4']['CLEAR_SNR'].items()}
THRESH_S4 = {k: float(v) for k, v in ann['step4']['THRESHOLD_SNR'].items()}

print('\nParameters loaded:')
for name, L, U in [
    ('Step 1 MIC',  MIC_L, MIC_U),
    ('Step 2 |ρ|',  RHO_L_S2, RHO_U_S2),
    ('Step 2 |r|',  R_L_S2, R_U_S2),
    ('Step 3 |ρ|',  RHO_L_S3, RHO_U_S3),
    ('Step 4 |r|',  R_L_S4, R_U_S4),
    ('Step 4 ΔBIC', BIC_L_S4, BIC_U_S4),
]:
    print(f'  {name:12s}  L={L:.3f}  U={U:.3f}')

In [ ]:
# ══════════════════════════════════════════════════════
#  计算所有标签 + 所有 membership
# ══════════════════════════════════════════════════════

# ── Step 1 labels ──
def label_s1(row):
    fid, snr = row['family_id'], row['snr_float']
    if fid == 'Null' or fid in NO_GLOBAL:
        return 0.0
    if fid not in CLEAR_S1:
        return np.nan
    if snr >= CLEAR_S1[fid]:
        return 1.0
    elif snr <= THRESH_S1[fid]:
        return 0.0
    return 0.5

data['label_s1'] = data.apply(label_s1, axis=1)
data['mu_s1'] = data['MIC'].apply(lambda v: fuzzy_mu(v, MIC_L, MIC_U))

# ── Step 2 labels ──
def label_s2(row):
    fid, snr = row['family_id'], row['snr_float']
    if fid in COMPLEX_FAM:
        return 0.0
    if fid in UNCERTAIN_FAM:
        if fid in CLEAR_S2 and snr >= CLEAR_S2[fid]:
            return 1.0
        if fid in THRESH_S2 and snr <= THRESH_S2[fid]:
            return 0.0
        return 0.5
    if fid not in CLEAR_S2:
        return np.nan
    if snr >= CLEAR_S2[fid]:
        return 1.0
    if fid in THRESH_S2 and snr <= THRESH_S2[fid]:
        return 0.0
    return 0.5

data['label_s2'] = data.apply(label_s2, axis=1)
data['mu_rho_s2'] = data['abs_rho'].apply(lambda v: fuzzy_mu(v, RHO_L_S2, RHO_U_S2))
data['mu_r_s2'] = data['abs_r'].apply(lambda v: fuzzy_mu(v, R_L_S2, R_U_S2))
data['mu_s2'] = np.minimum(data['mu_rho_s2'], data['mu_r_s2'])

# ── Step 3 labels ──
def label_s3(row):
    fid, snr = row['family_id'], row['snr_float']
    if fid in WEAK_MONO:
        return 0.0
    if fid in UNCERTAIN_MONO:
        return 0.5
    if fid not in CLEAR_S3:
        return np.nan
    if snr >= CLEAR_S3[fid]:
        return 1.0
    if fid in THRESH_S3 and snr <= THRESH_S3[fid]:
        return 0.0
    return 0.5

data['label_s3'] = data.apply(label_s3, axis=1)
data['mu_s3'] = data['abs_rho'].apply(lambda v: fuzzy_mu(v, RHO_L_S3, RHO_U_S3))

# ── Step 4 labels + ΔBIC ──
NONLINEAR_FAM = [f for f in FAMILIES if f not in LINEAR_FAM
                 and f not in NO_GLOBAL and f not in COMPLEX_FAM
                 and f not in WEAK_MONO
                 and f in (STRONG_MONO + UNCERTAIN_MONO)]

def label_s4(row):
    fid, snr = row['family_id'], row['snr_float']
    if fid not in LINEAR_FAM + NONLINEAR_FAM:
        return np.nan
    if fid in NONLINEAR_FAM:
        return 0.0
    if fid not in CLEAR_S4:
        return np.nan
    if snr >= CLEAR_S4[fid]:
        return 1.0
    if fid in THRESH_S4 and snr <= THRESH_S4[fid]:
        return 0.0
    return 0.5

data['label_s4'] = data.apply(label_s4, axis=1)
data['mu_r_s4'] = data['abs_r'].apply(lambda v: fuzzy_mu(v, R_L_S4, R_U_S4))

print('Computing ΔBIC for all cases (slow)...')
data['delta_bic'] = np.nan
for row_idx, row in data.iterrows():
    cid = row['case_id']
    if cid not in case_idx_map:
        continue
    idx = case_idx_map[cid]
    x, y = x_all[idx], y_all[idx]
    if len(x) < 10:
        continue
    data.at[row_idx, 'delta_bic'] = bic_compare(x, y)

data['mu_bic_s4'] = data['delta_bic'].apply(
    lambda v: 1.0 - fuzzy_mu(v, BIC_L_S4, BIC_U_S4) if pd.notna(v) else np.nan
)
data['mu_s4'] = np.minimum(data['mu_r_s4'], data['mu_bic_s4'])

print(f'ΔBIC computed: {data["delta_bic"].notna().sum()} cases')

# ── Step 5 labels + curvature features ──
def label_s5(row):
    fid = row['family_id']
    if fid not in CURVATURE_CLASS:
        return np.nan
    return CURVATURE_CLASS[fid]

data['label_s5_class'] = data.apply(label_s5, axis=1)
data['label_s5_curved'] = np.where(
    data['label_s5_class'].notna(), 1.0, np.nan
)

if 'step5_curvature' in P:
    print('Computing curvature features...')
    data['curv_strength'] = np.nan
    data['f2_dominance'] = np.nan
    data['f2_dom_sign'] = np.nan
    for row_idx, row in data.iterrows():
        cid = row['case_id']
        if cid not in case_idx_map:
            continue
        idx = case_idx_map[cid]
        x, y = x_all[idx], y_all[idx]
        if len(x) < 10:
            continue
        cs, fd, fds = curvature_evidence(x, y)
        data.at[row_idx, 'curv_strength'] = cs
        data.at[row_idx, 'f2_dominance'] = fd
        data.at[row_idx, 'f2_dom_sign'] = fds
    data['mu_s5_strength'] = data['curv_strength'].apply(
        lambda v: fuzzy_mu(v, C_L, C_U) if pd.notna(v) else np.nan
    )
    print(f'Curvature features computed: {data["curv_strength"].notna().sum()} cases')

print('\nLabel counts:')
for col in ['label_s1', 'label_s2', 'label_s3', 'label_s4']:
    vals = data[col].dropna()
    print(f'  {col}: Yes={int((vals==1).sum())} No={int((vals==0).sum())} Unc={int((vals==0.5).sum())}')

In [ ]:
# ══════════════════════════════════════════════════════
#  evaluate_gate_by_snr — 可复用评价函数
# ══════════════════════════════════════════════════════

def evaluate_gate_by_snr(
    data,
    mu_col,
    label_col,
    gate_name,
    alpha=0.5,
    n_bins=24,
):
    d = data[
        np.isfinite(data['snr_float'])
        & data[mu_col].notna()
        & data[label_col].notna()
    ].copy()

    d['_pred'] = (d[mu_col] >= alpha).astype(float)

    d['_correct'] = np.where(
        d[label_col].isin([0.0, 1.0]),
        (d['_pred'] == d[label_col]).astype(float),
        np.nan,
    )

    eps = 1e-9
    d['_model_uncertain'] = (
        (d[mu_col] > eps) & (d[mu_col] < 1 - eps)
    )

    log_snr = np.log10(d['snr_float'])
    edges = np.linspace(log_snr.min(), log_snr.max(), n_bins + 1)
    d['_snr_bin'] = pd.cut(log_snr, edges, include_lowest=True)

    grouped = d.groupby('_snr_bin', observed=True)

    summary = grouped.agg(
        snr=('snr_float', lambda x: 10 ** np.mean(np.log10(x))),
        correct_rate=('_correct', 'mean'),
        uncertain_rate=('_model_uncertain', 'mean'),
        n_cases=('case_id', 'size'),
    )

    positive_mu = (
        d[d[label_col] == 1]
        .groupby('_snr_bin', observed=True)[mu_col]
        .mean()
        .rename('mean_mu_positive')
    )

    negative_mu = (
        d[d[label_col] == 0]
        .groupby('_snr_bin', observed=True)[mu_col]
        .mean()
        .rename('mean_mu_negative')
    )

    summary = (
        summary
        .join(positive_mu)
        .join(negative_mu)
        .reset_index(drop=True)
        .sort_values('snr')
    )

    # ── 三张图 ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(
        summary['snr'], summary['correct_rate'],
        marker='o', markersize=3,
    )
    axes[0].axhline(0.9, color='gray', linestyle='--', label='90%')
    axes[0].set_title(f'{gate_name}: Correct rate')
    axes[0].set_ylabel('Correct classification rate')
    axes[0].legend()

    axes[1].plot(
        summary['snr'], summary['uncertain_rate'],
        marker='o', markersize=3, color='#E65100',
    )
    axes[1].set_title(f'{gate_name}: Uncertain rate')
    axes[1].set_ylabel('Fraction inside fuzzy range')

    axes[2].plot(
        summary['snr'], summary['mean_mu_positive'],
        marker='o', markersize=3, label='Human positive', color='#2e7d32',
    )
    axes[2].plot(
        summary['snr'], summary['mean_mu_negative'],
        marker='o', markersize=3, label='Human negative', color='#c62828',
    )
    axes[2].set_title(f'{gate_name}: Mean membership')
    axes[2].set_ylabel('Mean membership')
    axes[2].legend()

    for ax in axes:
        ax.set_xscale('log')
        ax.set_xlabel('SNR — log scale')
        ax.set_ylim(-0.03, 1.03)
        ax.grid(alpha=0.2)

    fig.suptitle(f'{gate_name} — {VARIANT} variant evaluation', fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    return summary


def reliable_snr_boundary(summary, threshold=0.90):
    s = summary.dropna(subset=['correct_rate']).sort_values('snr').copy()
    s['all_higher_above'] = (
        s['correct_rate'][::-1].cummin()[::-1].ge(threshold)
    )
    reliable = s.loc[s['all_higher_above'], 'snr']
    return reliable.min() if len(reliable) else np.nan


print('evaluate_gate_by_snr() 和 reliable_snr_boundary() 定义完成')

In [ ]:
# ══════════════════════════════════════════════════════
#  Step 1 — Global relationship (oracle)
# ══════════════════════════════════════════════════════

s1_summary = evaluate_gate_by_snr(
    data=data,
    mu_col='mu_s1',
    label_col='label_s1',
    gate_name='Step 1 — Global relationship',
    alpha=ALPHA,
)

s1_reliable = reliable_snr_boundary(s1_summary)
print(f'Step 1 reliable SNR boundary (≥90% correct): {s1_reliable:.2f}')

In [ ]:
# ══════════════════════════════════════════════════════
#  Step 2 — Simple relationship (oracle: label_s1 == 1)
# ══════════════════════════════════════════════════════

s2_eval = data[data['label_s1'] == 1].copy()
print(f'Step 2 oracle入口: {len(s2_eval)} cases (human Step 1 = Yes)')

s2_summary = evaluate_gate_by_snr(
    data=s2_eval,
    mu_col='mu_s2',
    label_col='label_s2',
    gate_name='Step 2 — Simple relationship',
    alpha=ALPHA,
)

s2_reliable = reliable_snr_boundary(s2_summary)
print(f'Step 2 reliable SNR boundary (≥90% correct): {s2_reliable:.2f}')

In [ ]:
# ══════════════════════════════════════════════════════
#  Step 3 — Strong monotonicity (oracle: S1=Yes, S2=Yes)
# ══════════════════════════════════════════════════════
# NOTE: Step 3-5 与 Step 1-2 的评价含义不同:
#   Step 1-2: SNR-based calibration — 标签由 SNR 驱动
#     ("在这个 SNR 下能不能看出来"), correct rate 下降 = 信号不可辨认
#   Step 3-5: Family-based classification — 标签由 family identity 决定
#     (F01 是 linear, F03 是 nonlinear), correct rate 下降 = 噪声让指标失真
#   两者都有意义: "在什么 SNR 下 gate 能正确区分 family 类型"

s3_eval = data[(data['label_s1'] == 1) & (data['label_s2'] == 1)].copy()
print(f'Step 3 oracle入口: {len(s3_eval)} cases (human S1=Yes, S2=Yes)')

s3_summary = evaluate_gate_by_snr(
    data=s3_eval,
    mu_col='mu_s3',
    label_col='label_s3',
    gate_name='Step 3 — Strong monotonicity',
    alpha=ALPHA,
)

s3_reliable = reliable_snr_boundary(s3_summary)
print(f'Step 3 reliable SNR boundary (≥90% correct): {s3_reliable:.2f}')

In [ ]:
# ══════════════════════════════════════════════════════
#  Step 4 — Linearity (oracle: S1=Yes, S2=Yes, S3=Yes)
# ══════════════════════════════════════════════════════

s4_eval = data[
    (data['label_s1'] == 1) &
    (data['label_s2'] == 1) &
    (data['label_s3'] == 1)
].copy()
print(f'Step 4 oracle入口: {len(s4_eval)} cases (human S1-S3=Yes)')

# Combined μ_line = min(μ_r, μ_bic)
s4_summary = evaluate_gate_by_snr(
    data=s4_eval,
    mu_col='mu_s4',
    label_col='label_s4',
    gate_name='Step 4 — Linearity (|r| + ΔBIC)',
    alpha=ALPHA,
)

s4_reliable = reliable_snr_boundary(s4_summary)
print(f'Step 4 reliable SNR boundary (≥90% correct): {s4_reliable:.2f}')

# Also show each sub-indicator separately
print('\n--- Sub-indicators ---')
s4r_summary = evaluate_gate_by_snr(
    data=s4_eval,
    mu_col='mu_r_s4',
    label_col='label_s4',
    gate_name='Step 4 sub — |r| only',
    alpha=ALPHA,
)

s4b_summary = evaluate_gate_by_snr(
    data=s4_eval.dropna(subset=['mu_bic_s4']),
    mu_col='mu_bic_s4',
    label_col='label_s4',
    gate_name='Step 4 sub — ΔBIC only',
    alpha=ALPHA,
)

In [ ]:
# ══════════════════════════════════════════════════════
#  Step 5 — Curvature (oracle: S1-S3=Yes, S4=Non-line)
# ══════════════════════════════════════════════════════

if 'step5_curvature' in P:
    s5_eval = data[
        (data['label_s1'] == 1) &
        (data['label_s2'] == 1) &
        (data['label_s3'] == 1) &
        (data['label_s4'] == 0)
    ].copy()
    print(f'Step 5 oracle入口: {len(s5_eval)} cases (human S1-S3=Yes, S4=Non-line)')

    s5_summary = evaluate_gate_by_snr(
        data=s5_eval.dropna(subset=['mu_s5_strength']),
        mu_col='mu_s5_strength',
        label_col='label_s5_curved',
        gate_name='Step 5 — Curvature strength',
        alpha=ALPHA,
    )

    s5_reliable = reliable_snr_boundary(s5_summary)
    print(f'Step 5 reliable SNR boundary (≥90% correct): {s5_reliable:.2f}')
else:
    print('Step 5 参数不存在，跳过')
    s5_reliable = np.nan

In [ ]:
# ══════════════════════════════════════════════════════
#  Summary — 各 gate 可靠 SNR 边界
# ══════════════════════════════════════════════════════

boundaries = pd.DataFrame([
    {'Gate': 'Step 1 — Global', 'Reliable SNR (≥90%)': s1_reliable,
     'L': MIC_L, 'U': MIC_U, 'Indicator': 'MIC'},
    {'Gate': 'Step 2 — Simple', 'Reliable SNR (≥90%)': s2_reliable,
     'L': f'{RHO_L_S2:.3f}/{R_L_S2:.3f}', 'U': f'{RHO_U_S2:.3f}/{R_U_S2:.3f}',
     'Indicator': 'min(|ρ|, |r|)'},
    {'Gate': 'Step 3 — Monotonic', 'Reliable SNR (≥90%)': s3_reliable,
     'L': RHO_L_S3, 'U': RHO_U_S3, 'Indicator': '|ρ|'},
    {'Gate': 'Step 4 — Linearity', 'Reliable SNR (≥90%)': s4_reliable,
     'L': f'{R_L_S4:.3f}/{BIC_L_S4:.1f}', 'U': f'{R_U_S4:.3f}/{BIC_U_S4:.1f}',
     'Indicator': 'min(|r|, 1-μ_BIC)'},
    {'Gate': 'Step 5 — Curvature', 'Reliable SNR (≥90%)': s5_reliable,
     'L': C_L if 'step5_curvature' in P else '-',
     'U': C_U if 'step5_curvature' in P else '-',
     'Indicator': 'curvature_strength'},
])

print('\n' + '=' * 70)
print(f'Fuzzy Decision Tree — {VARIANT} variant 各 gate 可靠 SNR 边界')
print('=' * 70)
display(boundaries)

print('\n解读:')
print('  Reliable SNR = 从该 SNR 起，所有更高 SNR 的正确率都 ≥ 90%')
print('  SNR > Reliable: 稳定正确')
print('  SNR ≈ Reliable: 不确定性最高')
print('  SNR < Reliable: 基本失效')

In [ ]:
# ══════════════════════════════════════════════════════
#  End-to-end cascade evaluation
#  (用算法预测路径，评价上游错误逐层传递的效果)
# ══════════════════════════════════════════════════════

# ── Algorithm predictions (cascading) ──
data['pred_s1'] = (data['mu_s1'] >= ALPHA).astype(float)
data['pred_s2'] = np.where(
    data['pred_s1'] == 1,
    (data['mu_s2'] >= ALPHA).astype(float),
    np.nan,
)
data['pred_s3'] = np.where(
    data['pred_s2'] == 1,
    (data['mu_s3'] >= ALPHA).astype(float),
    np.nan,
)
data['pred_s4'] = np.where(
    data['pred_s3'] == 1,
    (data['mu_s4'] >= ALPHA).astype(float),
    np.nan,
)

# ── End-to-end correct rate by SNR ──
def e2e_correct_by_snr(data, n_bins=24):
    d = data[np.isfinite(data['snr_float'])].copy()

    # Determine expected terminal type for each family
    EXPECTED = {}
    for fid in FAMILIES + ['Null']:
        if fid in NO_GLOBAL or fid == 'Null':
            EXPECTED[fid] = 'no_global'
        elif fid in COMPLEX_FAM:
            EXPECTED[fid] = 'complex'
        elif fid in WEAK_MONO:
            EXPECTED[fid] = 'weak_mono'
        elif fid in LINEAR_FAM:
            EXPECTED[fid] = 'linear'
        elif fid in CURVATURE_CLASS:
            EXPECTED[fid] = CURVATURE_CLASS[fid]
        else:
            EXPECTED[fid] = 'nonlinear'

    # Determine algorithm terminal type
    def algo_terminal(row):
        if row['pred_s1'] == 0:
            return 'no_global'
        if pd.isna(row['pred_s2']) or row['pred_s2'] == 0:
            return 'complex'
        if pd.isna(row['pred_s3']) or row['pred_s3'] == 0:
            return 'weak_mono'
        if pd.isna(row['pred_s4']):
            return 'unresolved'
        if row['pred_s4'] == 1:
            return 'linear'
        return 'nonlinear'

    d['expected_terminal'] = d['family_id'].map(EXPECTED)
    d['algo_terminal'] = d.apply(algo_terminal, axis=1)
    d['e2e_correct'] = d['expected_terminal'] == d['algo_terminal']

    log_snr = np.log10(d['snr_float'])
    edges = np.linspace(log_snr.min(), log_snr.max(), n_bins + 1)
    d['_snr_bin'] = pd.cut(log_snr, edges, include_lowest=True)

    grouped = d.groupby('_snr_bin', observed=True)
    summary = grouped.agg(
        snr=('snr_float', lambda x: 10 ** np.mean(np.log10(x))),
        e2e_correct_rate=('e2e_correct', 'mean'),
        n_cases=('case_id', 'size'),
    ).reset_index(drop=True).sort_values('snr')

    return d, summary

e2e_data, e2e_summary = e2e_correct_by_snr(data)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    e2e_summary['snr'], e2e_summary['e2e_correct_rate'],
    marker='o', markersize=4, color='#1565C0',
)
axes[0].axhline(0.9, color='gray', linestyle='--', label='90%')
axes[0].set_title('End-to-end correct rate vs SNR')
axes[0].set_ylabel('Correct rate')
axes[0].set_xscale('log')
axes[0].set_xlabel('SNR — log scale')
axes[0].set_ylim(-0.03, 1.03)
axes[0].grid(alpha=0.2)
axes[0].legend()

# Per-family success rate
fam_e2e = e2e_data.groupby('family_id').agg(
    correct_rate=('e2e_correct', 'mean'),
    n=('case_id', 'size'),
).reset_index().sort_values('family_id')

colors = ['#2e7d32' if r >= 0.9 else '#E65100' if r >= 0.7 else '#c62828'
          for r in fam_e2e['correct_rate']]
axes[1].barh(fam_e2e['family_id'], fam_e2e['correct_rate'], color=colors)
axes[1].axvline(0.9, color='gray', linestyle='--')
axes[1].set_title('End-to-end correct rate by family')
axes[1].set_xlabel('Correct rate')
axes[1].set_xlim(0, 1.05)
axes[1].invert_yaxis()

fig.suptitle(f'End-to-end cascade — {VARIANT} variant', fontweight='bold')
plt.tight_layout()
plt.show()
plt.close(fig)

e2e_reliable = reliable_snr_boundary(e2e_summary.rename(columns={'e2e_correct_rate': 'correct_rate'}))
print(f'\nEnd-to-end reliable SNR boundary (≥90%): {e2e_reliable:.2f}')
print(f'Overall correct rate: {e2e_data["e2e_correct"].mean():.1%}')
print()
display(fam_e2e)

In [ ]:
# ══════════════════════════════════════════════════════
#  Per-family SNR 曲线 — 每个 family 单独画 correct rate vs SNR
# ══════════════════════════════════════════════════════

e2e_families = sorted(e2e_data['family_id'].unique())
n_fam = len(e2e_families)
ncols = 6
nrows = int(np.ceil(n_fam / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3 * nrows))
axes_flat = axes.flatten()

for i, fid in enumerate(e2e_families):
    ax = axes_flat[i]
    fd = e2e_data[e2e_data['family_id'] == fid].copy()
    if len(fd) < 5:
        ax.set_title(f'{fid} (n={len(fd)})', fontsize=9)
        continue

    log_snr = np.log10(fd['snr_float'])
    edges = np.linspace(log_snr.min(), log_snr.max(), 13)
    fd['_bin'] = pd.cut(log_snr, edges, include_lowest=True)
    grp = fd.groupby('_bin', observed=True).agg(
        snr=('snr_float', lambda x: 10 ** np.mean(np.log10(x))),
        correct_rate=('e2e_correct', 'mean'),
    ).reset_index(drop=True).sort_values('snr')

    overall = fd['e2e_correct'].mean()
    color = '#2e7d32' if overall >= 0.9 else '#E65100' if overall >= 0.7 else '#c62828'
    ax.plot(grp['snr'], grp['correct_rate'], marker='o', markersize=3, color=color)
    ax.axhline(0.9, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xscale('log')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f'{fid} ({overall:.0%})', fontsize=9, color=color, fontweight='bold')
    ax.grid(alpha=0.15)
    if i % ncols == 0:
        ax.set_ylabel('Correct rate', fontsize=8)
    if i >= n_fam - ncols:
        ax.set_xlabel('SNR', fontsize=8)

for j in range(n_fam, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(f'Per-family end-to-end correct rate vs SNR — {VARIANT}',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()
plt.close(fig)